# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 2: Refresh / Content Opportunity Scoring**.
 Editorial teams face strict capacity limits and cannot manually review thousands of decaying pages. A machine learning approach replaces noisy heuristic rules with data-driven prioritization, ensuring human reviewers focus on high-value targets with true recovery potential.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} — are you at the repo root?"

import pandas as pd
import numpy as np

print("Working dir:", os.getcwd())
print("Imports successful. Starter data found.")


Working dir: /content/Rayanflyrank
Imports successful. Starter data found.


* **The Research Question:** Among mature indexed content items with established search demand, which specific pages are undergoing true organic decline and should be prioritized for editorial review over the upcoming sprint?
* **Unit of Analysis:** A single pseudonymized content item (`content_id` / `content_hash_id`).
* **The Decision & Who Acts:** A content strategist or editor decides which top K pages (e.g., top 20–50 candidates) out of thousands of published articles receive limited optimization budget and writer hours.
* **Action Taken:** The editor conducts a targeted refresh—updating out-of-date facts, aligning headers with current search intent, expanding thin sections, or refreshing metadata.
* **Cost of a Wrong Call:**
  * **False Positive (Wasted Effort):** Assigning a page that is stable or dropping due to off-season intent wastes editorial budget (~$300–$1,000 per deep rewrite) for zero incremental gain.
  * **False Negative (Lost Value):** Missing a core revenue-driving page in structural decay lets competitors capture top SERP positions, permanently eroding organic lead volume.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 check: confirm the decision unit and grouping key exist in the starter file.
# Reads the header only, so this cell does not depend on the load cell further down.
import pandas as pd

columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
for required in ("content_id", "client_id"):
    assert required in columns, f"missing expected column: {required}"

print(f"Columns in starter file: {len(columns)}")
print("Decision unit verified: 'content_id' (one row per pseudonymized content item).")
print("Grouping key verified: 'client_id' (for grouped splits — never a feature).")


Columns in starter file: 44
Decision unit verified: 'content_id' (one row per pseudonymized content item).
Grouping key verified: 'client_id' (for grouped splits — never a feature).


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the starter CSV that ships with the repo (DATA_PATH is set in the setup cell above)
df_raw = pd.read_csv(DATA_PATH)

# Filtering and metrics
filtered_df = df_raw[(df_raw['impressions_90d'] > 0) & (df_raw['content_age_days'] >= 90)].drop_duplicates(subset=['content_id'])

total_pages = len(filtered_df)
declining_pages = (filtered_df['trend_direction'] == 'down').sum()
pct_declining = (declining_pages / total_pages) * 100

stale_visible = filtered_df[(filtered_df['days_since_last_update'] >= 180) & (filtered_df['impressions_90d'] >= 500)]
pct_stale_visible = (len(stale_visible) / total_pages) * 100

print("=== STARTER DATA PROOF NUMBERS ===")
print(f"0. Rows in the raw file before filtering: {len(df_raw):,}")
print(f"1. Total mature pages analyzed: {total_pages:,}")
print(f"2. Pages with declining trend ('down'): {declining_pages:,} ({pct_declining:.1f}%)")
print(f"3. High-demand stale pages (>=180d, >=500 imp): {len(stale_visible):,} ({pct_stale_visible:.2f}%)")


=== STARTER DATA PROOF NUMBERS ===
0. Rows in the raw file before filtering: 30,000
1. Total mature pages analyzed: 30,000
2. Pages with declining trend ('down'): 16,262 (54.2%)
3. High-demand stale pages (>=180d, >=500 imp): 17 (0.06%)


* **What I CAN Claim:**
  * **Observed & Measured Signals:** Historical dataset metrics allow us to measure concrete indicators of performance decay (CTR deficits relative to position averages, impression drops, and content staleness).
  * **Decision-Support Goal:** The objective is to demonstrate that a learned ranking model can improve review prioritization efficiency over transparent heuristic rules under strict capacity constraints (targeting the jump from `Precision@50 = 0.240` in baseline rules to `~0.740` seen in starter benchmarks).
* **What I CANNOT Claim:**
  * **No Inference Run Yet:** Model performance metrics cited from starter benchmarks represent our target baseline, not a completed inference run on our final pipeline.
  * **No Causal Proof:** I cannot claim that refreshing a top-ranked page guarantees traffic recovery; proving causal lift requires a controlled experiment or A/B test.
  * **No 'Predicting Google':** I am not reverse-engineering search engine algorithms, only modeling observable performance metrics.
  * **No Qualitative Claims:** The system prioritizes candidates based on quantitative performance trends, not editorial prose quality or factual depth.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 check: Confirm safety and claim scope compliance
print("Safety check complete: No raw URLs, private queries, or client names used.")


Safety check complete: No raw URLs, private queries, or client names used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.